# HW12 - временные ряды: temporal split, baseline-модели и GRU

В ноутбуке собраны загрузка данных, корректный `temporal split`, признаки для baseline-моделей, оконное представление для `GRU`, сравнение экспериментов `B1`, `B2`, `B3`, `R1` и сохранение обязательных артефактов.

## 0. Импорты, seed и служебные функции

Вся логика домашней работы находится прямо в этом ноутбуке: от подготовки данных до обучения `GRU` и сохранения артефактов.

In [1]:
from __future__ import annotations

import json
import math
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset


SEED = 42
DATASET_NAME = "S12-hw-dataset.csv"
TARGET_COL = "target"
DATE_COL = "date"
HORIZON = 1
MOVING_AVG_WINDOW = 24
WINDOW_SIZE = 48
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15


@dataclass
class ProjectPaths:
    base_dir: Path
    data_path: Path
    artifacts_dir: Path
    figures_dir: Path
    runs_path: Path
    best_gru_path: Path
    best_gru_config_path: Path

    @classmethod
    def from_base_dir(cls, base_dir: Path) -> "ProjectPaths":
        artifacts_dir = base_dir / "artifacts"
        figures_dir = artifacts_dir / "figures"
        return cls(
            base_dir=base_dir,
            data_path=base_dir / "data" / DATASET_NAME,
            artifacts_dir=artifacts_dir,
            figures_dir=figures_dir,
            runs_path=artifacts_dir / "runs.csv",
            best_gru_path=artifacts_dir / "best_gru.pt",
            best_gru_config_path=artifacts_dir / "best_gru_config.json",
        )


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"


def ensure_dirs(paths: ProjectPaths) -> None:
    paths.artifacts_dir.mkdir(parents=True, exist_ok=True)
    paths.figures_dir.mkdir(parents=True, exist_ok=True)


def load_dataset(data_path: Path) -> pd.DataFrame:
    df = pd.read_csv(data_path)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    df = df.sort_values(DATE_COL).reset_index(drop=True)
    return df


def detect_frequency(df: pd.DataFrame) -> str:
    freq = pd.infer_freq(df[DATE_COL])
    if freq:
        return freq
    deltas = df[DATE_COL].diff().dropna()
    if deltas.empty:
        return "unknown"
    mode_delta = deltas.mode().iloc[0]
    if mode_delta == pd.Timedelta(hours=1):
        return "hourly"
    return str(mode_delta)


def summarize_splits(df: pd.DataFrame) -> dict[str, slice]:
    n_obs = len(df)
    train_end = int(n_obs * TRAIN_RATIO)
    val_end = int(n_obs * (TRAIN_RATIO + VAL_RATIO))
    return {
        "train": slice(0, train_end),
        "validation": slice(train_end, val_end),
        "test": slice(val_end, n_obs),
    }


def split_boundaries(df: pd.DataFrame, splits: dict[str, slice]) -> dict[str, dict[str, Any]]:
    summary: dict[str, dict[str, Any]] = {}
    for name, slc in splits.items():
        part = df.iloc[slc]
        summary[name] = {
            "start": part[DATE_COL].iloc[0],
            "end": part[DATE_COL].iloc[-1],
            "size": len(part),
        }
    return summary


def plot_series_split(df: pd.DataFrame, splits: dict[str, slice], output_path: Path) -> None:
    plt.figure(figsize=(14, 5))
    colors = {"train": "#2a9d8f", "validation": "#e9c46a", "test": "#e76f51"}
    labels = {"train": "Train", "validation": "Validation", "test": "Test"}
    for key, slc in splits.items():
        part = df.iloc[slc]
        plt.plot(part[DATE_COL], part[TARGET_COL], label=labels[key], color=colors[key], linewidth=1.2)
    plt.title("Temporal Split For Forecasting")
    plt.xlabel("Date")
    plt.ylabel("Target")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


def make_feature_frame(df: pd.DataFrame) -> pd.DataFrame:
    features = df.copy()
    for lag in [1, 7, 14, 24, 168]:
        features[f"lag_{lag}"] = features[TARGET_COL].shift(lag)

    shifted = features[TARGET_COL].shift(1)
    features["rolling_mean_7"] = shifted.rolling(7).mean()
    features["rolling_std_7"] = shifted.rolling(7).std()
    features["rolling_mean_24"] = shifted.rolling(24).mean()
    features["rolling_std_24"] = shifted.rolling(24).std()

    features["day_of_week"] = features[DATE_COL].dt.dayofweek
    features["hour"] = features[DATE_COL].dt.hour
    features["is_weekend"] = (features["day_of_week"] >= 5).astype(int)

    return features


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    safe_true = np.where(np.abs(y_true) < 1e-8, 1e-8, y_true)
    mape = np.mean(np.abs((y_true - y_pred) / safe_true)) * 100.0
    return {"mae": float(mae), "rmse": float(rmse), "mape": float(mape)}


def baseline_naive(df: pd.DataFrame, split_slice: slice) -> tuple[np.ndarray, np.ndarray, pd.Series]:
    part = df.iloc[split_slice]
    y_true = part[TARGET_COL].to_numpy()
    y_pred = df[TARGET_COL].shift(1).iloc[split_slice].to_numpy()
    return y_true, y_pred, part[DATE_COL]


def baseline_moving_average(df: pd.DataFrame, split_slice: slice, window: int = MOVING_AVG_WINDOW) -> tuple[np.ndarray, np.ndarray, pd.Series]:
    rolling_pred = df[TARGET_COL].shift(1).rolling(window).mean()
    part = df.iloc[split_slice]
    return part[TARGET_COL].to_numpy(), rolling_pred.iloc[split_slice].to_numpy(), part[DATE_COL]


def ridge_with_lag_features(
    feature_df: pd.DataFrame,
    splits: dict[str, slice],
) -> dict[str, Any]:
    feature_cols = [
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_24",
        "lag_168",
        "rolling_mean_7",
        "rolling_std_7",
        "rolling_mean_24",
        "rolling_std_24",
        "day_of_week",
        "hour",
        "is_weekend",
    ]
    ready = feature_df.dropna().reset_index(drop=True)
    train_end = splits["train"].stop
    val_end = splits["validation"].stop

    train_df = ready.iloc[: max(train_end - 168, 1)]
    val_df = ready.iloc[max(train_end - 168, 1) : max(val_end - 168, 1)]
    test_df = ready.iloc[max(val_end - 168, 1) :]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[feature_cols])
    X_val = scaler.transform(val_df[feature_cols])
    X_test = scaler.transform(test_df[feature_cols])

    model = Ridge(alpha=1.0)
    model.fit(X_train, train_df[TARGET_COL])

    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)

    return {
        "model": model,
        "scaler": scaler,
        "feature_cols": feature_cols,
        "val_true": val_df[TARGET_COL].to_numpy(),
        "val_pred": val_pred,
        "val_dates": val_df[DATE_COL],
        "test_true": test_df[TARGET_COL].to_numpy(),
        "test_pred": test_pred,
        "test_dates": test_df[DATE_COL],
    }


class SequenceDataset(Dataset):
    def __init__(self, values: np.ndarray, indices: list[int], window_size: int) -> None:
        self.values = values
        self.indices = indices
        self.window_size = window_size

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        target_idx = self.indices[idx]
        start = target_idx - self.window_size
        window = self.values[start:target_idx]
        target = self.values[target_idx]
        return (
            torch.tensor(window[:, None], dtype=torch.float32),
            torch.tensor([target], dtype=torch.float32),
        )


class GRUForecast(nn.Module):
    def __init__(self, input_size: int = 1, hidden_size: int = 32, num_layers: int = 1, dropout: float = 0.0) -> None:
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.gru(x)
        last_hidden = out[:, -1, :]
        return self.head(last_hidden)


def predict_with_gru(
    model: nn.Module,
    values_scaled: np.ndarray,
    indices: list[int],
    window_size: int,
    scaler: StandardScaler,
    device: str,
) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for idx in indices:
            window = values_scaled[idx - window_size : idx]
            x = torch.tensor(window[None, :, None], dtype=torch.float32, device=device)
            pred_scaled = model(x).cpu().numpy().reshape(-1, 1)
            true_scaled = np.array([[values_scaled[idx]]])
            preds.append(scaler.inverse_transform(pred_scaled)[0, 0])
            trues.append(scaler.inverse_transform(true_scaled)[0, 0])
    return np.asarray(trues), np.asarray(preds)


def train_gru_model(
    df: pd.DataFrame,
    splits: dict[str, slice],
    paths: ProjectPaths,
    seed: int = SEED,
    window_size: int = WINDOW_SIZE,
    hidden_size: int = 32,
    batch_size: int = 64,
    lr: float = 1e-3,
    max_epochs: int = 40,
) -> dict[str, Any]:
    set_seed(seed)
    device = get_device()
    values = df[TARGET_COL].to_numpy(dtype=np.float32)
    train_end = splits["train"].stop
    val_end = splits["validation"].stop

    scaler = StandardScaler()
    scaler.fit(values[:train_end].reshape(-1, 1))
    values_scaled = scaler.transform(values.reshape(-1, 1)).reshape(-1)

    train_indices = list(range(window_size, train_end))
    val_indices = list(range(train_end, val_end))
    test_indices = list(range(val_end, len(df)))

    train_loader = DataLoader(
        SequenceDataset(values_scaled, train_indices, window_size),
        batch_size=batch_size,
        shuffle=True,
    )

    model = GRUForecast(hidden_size=hidden_size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    history: list[dict[str, float]] = []
    best_val_mae = float("inf")
    best_state: dict[str, Any] | None = None
    best_epoch = 0
    best_val_metrics: dict[str, float] | None = None

    for epoch in range(1, max_epochs + 1):
        model.train()
        batch_losses = []
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            preds = model(x_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())

        train_loss = float(np.mean(batch_losses))
        val_true, val_pred = predict_with_gru(model, values_scaled, val_indices, window_size, scaler, device)
        val_metrics = compute_metrics(val_true, val_pred)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_mae": val_metrics["mae"],
                "val_rmse": val_metrics["rmse"],
                "val_mape": val_metrics["mape"],
            }
        )

        if val_metrics["mae"] < best_val_mae:
            best_val_mae = val_metrics["mae"]
            best_epoch = epoch
            best_val_metrics = val_metrics
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

    if best_state is None or best_val_metrics is None:
        raise RuntimeError("GRU training did not produce a valid best state.")

    torch.save(best_state, paths.best_gru_path)
    config = {
        "seed": seed,
        "window_size": window_size,
        "hidden_size": hidden_size,
        "batch_size": batch_size,
        "learning_rate": lr,
        "max_epochs": max_epochs,
        "optimizer": "Adam",
        "loss": "MSELoss",
        "scaler": "StandardScaler(train target only)",
        "device": device,
        "best_epoch": best_epoch,
    }
    paths.best_gru_config_path.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")

    best_model = GRUForecast(hidden_size=hidden_size).to(device)
    best_model.load_state_dict(best_state)

    val_true, val_pred = predict_with_gru(best_model, values_scaled, val_indices, window_size, scaler, device)
    test_true, test_pred = predict_with_gru(best_model, values_scaled, test_indices, window_size, scaler, device)

    return {
        "model": best_model,
        "history": pd.DataFrame(history),
        "config": config,
        "val_true": val_true,
        "val_pred": val_pred,
        "val_dates": df.iloc[val_indices][DATE_COL],
        "test_true": test_true,
        "test_pred": test_pred,
        "test_dates": df.iloc[test_indices][DATE_COL],
        "val_metrics": compute_metrics(val_true, val_pred),
        "test_metrics": compute_metrics(test_true, test_pred),
    }


def plot_learning_curves(history: pd.DataFrame, output_path: Path) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    axes[0].plot(history["epoch"], history["train_loss"], color="#264653")
    axes[0].set_title("GRU Train Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")

    axes[1].plot(history["epoch"], history["val_mae"], label="Val MAE", color="#e76f51")
    axes[1].plot(history["epoch"], history["val_rmse"], label="Val RMSE", color="#2a9d8f")
    axes[1].set_title("Validation Metrics")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def plot_model_comparison(results_df: pd.DataFrame, output_path: Path) -> None:
    plt.figure(figsize=(9, 5))
    ordered = results_df.sort_values("best_val_mae")
    plt.bar(ordered["experiment_id"], ordered["best_val_mae"], color=["#577590", "#43aa8b", "#f9c74f", "#f3722c"])
    plt.title("Validation MAE By Experiment")
    plt.xlabel("Experiment")
    plt.ylabel("MAE")
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


def plot_forecast(dates: pd.Series, y_true: np.ndarray, y_pred: np.ndarray, title: str, output_path: Path) -> None:
    plt.figure(figsize=(14, 5))
    plt.plot(dates, y_true, label="Actual", color="#264653", linewidth=1.3)
    plt.plot(dates, y_pred, label="Forecast", color="#e76f51", linewidth=1.2)
    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Target")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


def describe_series(df: pd.DataFrame) -> str:
    trend = df[TARGET_COL].iloc[-1] - df[TARGET_COL].iloc[0]
    daily_autocorr = df[TARGET_COL].autocorr(lag=24)
    weekly_autocorr = df[TARGET_COL].autocorr(lag=24 * 7)
    volatility = df[TARGET_COL].diff().abs().quantile(0.95)
    parts = []
    if trend > 0:
        parts.append("ряд показывает умеренный восходящий тренд")
    else:
        parts.append("ряд не демонстрирует выраженного роста и скорее колеблется вокруг среднего")
    if pd.notna(daily_autocorr) and daily_autocorr > 0.6:
        parts.append("хорошо заметна дневная сезонность")
    if pd.notna(weekly_autocorr) and weekly_autocorr > 0.4:
        parts.append("виден и недельный повторяющийся паттерн")
    parts.append(f"крупные почасовые скачки доходят примерно до {volatility:.2f}")
    parts.append("это указывает на нестационарность уровня и дисперсии, поэтому temporal split обязателен")
    return ", ".join(parts) + "."


def build_runs_table(
    split_summary_text: str,
    ridge_result: dict[str, Any],
    gru_result: dict[str, Any],
    baseline_metrics: dict[str, dict[str, float]],
    best_experiment: str,
    best_test_metrics: dict[str, float],
) -> pd.DataFrame:
    rows = [
        {
            "experiment_id": "B1",
            "task": "forecasting",
            "dataset": DATASET_NAME,
            "seed": SEED,
            "split_summary": split_summary_text,
            "window_size": "",
            "horizon": HORIZON,
            "model_summary": "naive-last",
            "features_summary": "last observed target",
            "scaler": "",
            "optimizer": "",
            "lr": "",
            "epochs_trained": "",
            "best_val_mae": baseline_metrics["B1"]["mae"],
            "best_val_rmse": baseline_metrics["B1"]["rmse"],
            "best_val_mape": baseline_metrics["B1"]["mape"],
            "test_mae": best_test_metrics["mae"] if best_experiment == "B1" else "",
            "test_rmse": best_test_metrics["rmse"] if best_experiment == "B1" else "",
            "test_mape": best_test_metrics["mape"] if best_experiment == "B1" else "",
            "notes": "No training required",
        },
        {
            "experiment_id": "B2",
            "task": "forecasting",
            "dataset": DATASET_NAME,
            "seed": SEED,
            "split_summary": split_summary_text,
            "window_size": MOVING_AVG_WINDOW,
            "horizon": HORIZON,
            "model_summary": f"moving-average-{MOVING_AVG_WINDOW}",
            "features_summary": "rolling mean of target using past values only",
            "scaler": "",
            "optimizer": "",
            "lr": "",
            "epochs_trained": "",
            "best_val_mae": baseline_metrics["B2"]["mae"],
            "best_val_rmse": baseline_metrics["B2"]["rmse"],
            "best_val_mape": baseline_metrics["B2"]["mape"],
            "test_mae": best_test_metrics["mae"] if best_experiment == "B2" else "",
            "test_rmse": best_test_metrics["rmse"] if best_experiment == "B2" else "",
            "test_mape": best_test_metrics["mape"] if best_experiment == "B2" else "",
            "notes": "No training required",
        },
        {
            "experiment_id": "B3",
            "task": "forecasting",
            "dataset": DATASET_NAME,
            "seed": SEED,
            "split_summary": split_summary_text,
            "window_size": "",
            "horizon": HORIZON,
            "model_summary": "Ridge(alpha=1.0)",
            "features_summary": ", ".join(ridge_result["feature_cols"]),
            "scaler": "StandardScaler(features, fit on train)",
            "optimizer": "",
            "lr": "",
            "epochs_trained": "",
            "best_val_mae": ridge_result["val_metrics"]["mae"],
            "best_val_rmse": ridge_result["val_metrics"]["rmse"],
            "best_val_mape": ridge_result["val_metrics"]["mape"],
            "test_mae": best_test_metrics["mae"] if best_experiment == "B3" else "",
            "test_rmse": best_test_metrics["rmse"] if best_experiment == "B3" else "",
            "test_mape": best_test_metrics["mape"] if best_experiment == "B3" else "",
            "notes": "Lag, rolling and calendar features without future leakage",
        },
        {
            "experiment_id": "R1",
            "task": "forecasting",
            "dataset": DATASET_NAME,
            "seed": SEED,
            "split_summary": split_summary_text,
            "window_size": gru_result["config"]["window_size"],
            "horizon": HORIZON,
            "model_summary": f"GRU(hidden_size={gru_result['config']['hidden_size']}, layers=1)",
            "features_summary": "past target sequence only",
            "scaler": gru_result["config"]["scaler"],
            "optimizer": gru_result["config"]["optimizer"],
            "lr": gru_result["config"]["learning_rate"],
            "epochs_trained": gru_result["config"]["best_epoch"],
            "best_val_mae": gru_result["val_metrics"]["mae"],
            "best_val_rmse": gru_result["val_metrics"]["rmse"],
            "best_val_mape": gru_result["val_metrics"]["mape"],
            "test_mae": best_test_metrics["mae"] if best_experiment == "R1" else "",
            "test_rmse": best_test_metrics["rmse"] if best_experiment == "R1" else "",
            "test_mape": best_test_metrics["mape"] if best_experiment == "R1" else "",
            "notes": "Best checkpoint selected by validation MAE",
        },
    ]
    return pd.DataFrame(rows)


def run_homework(base_dir: str | Path) -> dict[str, Any]:
    base_path = Path(base_dir)
    paths = ProjectPaths.from_base_dir(base_path)
    ensure_dirs(paths)
    set_seed(SEED)

    df = load_dataset(paths.data_path)
    splits = summarize_splits(df)
    boundaries = split_boundaries(df, splits)
    split_summary_text = (
        f"train={boundaries['train']['start']}..{boundaries['train']['end']} ({boundaries['train']['size']}), "
        f"val={boundaries['validation']['start']}..{boundaries['validation']['end']} ({boundaries['validation']['size']}), "
        f"test={boundaries['test']['start']}..{boundaries['test']['end']} ({boundaries['test']['size']})"
    )

    plot_series_split(df, splits, paths.figures_dir / "series_split.png")

    y_val_b1, pred_val_b1, val_dates_b1 = baseline_naive(df, splits["validation"])
    y_test_b1, pred_test_b1, test_dates_b1 = baseline_naive(df, splits["test"])
    y_val_b2, pred_val_b2, val_dates_b2 = baseline_moving_average(df, splits["validation"])
    y_test_b2, pred_test_b2, test_dates_b2 = baseline_moving_average(df, splits["test"])

    baseline_metrics = {
        "B1": compute_metrics(y_val_b1, pred_val_b1),
        "B2": compute_metrics(y_val_b2, pred_val_b2),
    }

    feature_df = make_feature_frame(df)
    ridge_result = ridge_with_lag_features(feature_df, splits)
    ridge_result["val_metrics"] = compute_metrics(ridge_result["val_true"], ridge_result["val_pred"])
    ridge_result["test_metrics"] = compute_metrics(ridge_result["test_true"], ridge_result["test_pred"])

    gru_result = train_gru_model(df, splits, paths)
    plot_learning_curves(gru_result["history"], paths.figures_dir / "gru_learning_curves.png")

    val_scoreboard = {
        "B1": baseline_metrics["B1"]["mae"],
        "B2": baseline_metrics["B2"]["mae"],
        "B3": ridge_result["val_metrics"]["mae"],
        "R1": gru_result["val_metrics"]["mae"],
    }
    best_experiment = min(val_scoreboard, key=val_scoreboard.get)

    test_lookup = {
        "B1": {
            "metrics": compute_metrics(y_test_b1, pred_test_b1),
            "dates": test_dates_b1,
            "y_true": y_test_b1,
            "y_pred": pred_test_b1,
        },
        "B2": {
            "metrics": compute_metrics(y_test_b2, pred_test_b2),
            "dates": test_dates_b2,
            "y_true": y_test_b2,
            "y_pred": pred_test_b2,
        },
        "B3": {
            "metrics": ridge_result["test_metrics"],
            "dates": ridge_result["test_dates"],
            "y_true": ridge_result["test_true"],
            "y_pred": ridge_result["test_pred"],
        },
        "R1": {
            "metrics": gru_result["test_metrics"],
            "dates": gru_result["test_dates"],
            "y_true": gru_result["test_true"],
            "y_pred": gru_result["test_pred"],
        },
    }

    best_test = test_lookup[best_experiment]
    plot_forecast(
        best_test["dates"],
        best_test["y_true"],
        best_test["y_pred"],
        f"Best Model Forecast On Test ({best_experiment})",
        paths.figures_dir / "best_forecast_test.png",
    )

    runs_df = build_runs_table(
        split_summary_text=split_summary_text,
        ridge_result=ridge_result,
        gru_result=gru_result,
        baseline_metrics=baseline_metrics,
        best_experiment=best_experiment,
        best_test_metrics=best_test["metrics"],
    )
    runs_df.to_csv(paths.runs_path, index=False)
    plot_model_comparison(runs_df, paths.figures_dir / "baselines_compare.png")

    context = {
        "paths": asdict(paths),
        "seed": SEED,
        "device": get_device(),
        "frequency": detect_frequency(df),
        "series_comment": describe_series(df),
        "df": df,
        "splits": splits,
        "boundaries": boundaries,
        "feature_df": feature_df,
        "baseline_metrics": baseline_metrics,
        "ridge_result": ridge_result,
        "gru_result": gru_result,
        "runs_df": runs_df,
        "best_experiment": best_experiment,
        "best_test_metrics": best_test["metrics"],
        "best_test_payload": best_test,
        "random_split_reason": "Random split mixes past and future observations, so the model would validate on points that are chronologically earlier than parts of the training data and receive an unrealistic advantage from temporal leakage.",
        "leakage_note": "Утечка могла бы возникнуть, если строить лаги или rolling-статистики с текущим значением, а также если обучать scaler на всём датасете вместо train.",
    }
    return context


from pathlib import Path
BASE_DIR = Path.cwd()
paths = ProjectPaths.from_base_dir(BASE_DIR)
set_seed(SEED)
device = get_device()
print({'device': device, 'seed': SEED})

{'device': 'cpu', 'seed': 42}


## 1. Данные и первичный анализ

Загружаем `S12-hw-dataset.csv`, приводим колонку `date` к `datetime`, сортируем по времени и делаем sanity-check.

In [2]:
df = load_dataset(paths.data_path)
display(df.head())
print('shape:', df.shape)
print('date range:', df['date'].min(), '->', df['date'].max())
print('missing values:')
display(df.isna().sum().to_frame('missing'))

plt.figure(figsize=(14, 4))
plt.plot(df['date'], df['target'], color='#264653', linewidth=1)
plt.title('Исходный временной ряд')
plt.xlabel('date')
plt.ylabel('target')
plt.tight_layout()
plt.show()

print(describe_series(df))

,date,target
0,2025-01-01 00:00:00,98.14
1,2025-01-01 01:00:00,98.07
2,2025-01-01 02:00:00,104.70
3,2025-01-01 03:00:00,112.81
4,2025-01-01 04:00:00,112.62


shape: (4320, 2)
date range: 2025-01-01 00:00:00 -> 2025-06-29 23:00:00
missing values:


,missing
date,0
target,0


ряд показывает умеренный восходящий тренд, хорошо заметна дневная сезонность, виден и недельный повторяющийся паттерн, крупные почасовые скачки доходят примерно до 16.10, это указывает на нестационарность уровня и дисперсии, поэтому temporal split обязателен.


/tmp/ipykernel_53737/281571060.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Ряд почасовой, а наблюдения зависят от порядка во времени. Поэтому использовать `random split` здесь нельзя: он перемешал бы прошлое и будущее и привёл бы к оптимистичной оценке качества.

## 2. Корректный Temporal Split

In [3]:
splits = summarize_splits(df)
boundaries = split_boundaries(df, splits)
display(pd.DataFrame(boundaries).T)
plot_series_split(df, splits, paths.figures_dir / 'series_split.png')
img = plt.imread(paths.figures_dir / 'series_split.png')
plt.figure(figsize=(14, 5))
plt.imshow(img)
plt.axis('off')
plt.show()

,start,end,size
train,2025-01-01 00:00:00,2025-05-06 23:00:00,3024
validation,2025-05-07 00:00:00,2025-06-02 23:00:00,648
test,2025-06-03 00:00:00,2025-06-29 23:00:00,648


/tmp/ipykernel_53737/4090509048.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


В `train` попадает только прошлое, в `validation` следующий непрерывный интервал, а `test` зарезервирован для одной финальной проверки лучшего подхода.

## 3. Признаки для baseline-моделей

Строим лаговые, rolling- и календарные признаки без утечки информации из будущего.

In [4]:
feature_df = make_feature_frame(df)
display(feature_df[['date', 'target', 'lag_1', 'lag_7', 'lag_14', 'rolling_mean_7', 'rolling_std_7', 'day_of_week', 'hour']].head(20))

,date,target,lag_1,lag_7,lag_14,rolling_mean_7,rolling_std_7,day_of_week,hour
0,2025-01-01 00:00:00,98.14,NaN,NaN,NaN,NaN,NaN,2,0
1,2025-01-01 01:00:00,98.07,98.14,NaN,NaN,NaN,NaN,2,1
2,2025-01-01 02:00:00,104.70,98.07,NaN,NaN,NaN,NaN,2,2
3,2025-01-01 03:00:00,112.81,104.70,NaN,NaN,NaN,NaN,2,3
4,2025-01-01 04:00:00,112.62,112.81,NaN,NaN,NaN,NaN,2,4
5,2025-01-01 05:00:00,117.29,112.62,NaN,NaN,NaN,NaN,2,5
6,2025-01-01 06:00:00,126.50,117.29,NaN,NaN,NaN,NaN,2,6
7,2025-01-01 07:00:00,126.01,126.50,98.14,NaN,110.018571,10.418147,2,7
8,2025-01-01 08:00:00,122.38,126.01,98.07,NaN,114.000000,10.447405,2,8
9,2025-01-01 09:00:00,128.28,122.38,104.70,NaN,117.472857,8.030400,2,9


Обязательные признаки: `lag_1`, `lag_7`, `lag_14`, `rolling_mean_7`, `rolling_std_7`, `day_of_week`. Дополнительно использованы `hour`, `is_weekend`, а также более длинные лаги `24` и `168` часов.

## 4. Оконное представление для GRU

Для нейросети ряд переводится в последовательности фиксированной длины `window_size=48`, после чего модель прогнозирует следующее значение.

In [5]:
context = run_homework(BASE_DIR)
display(context['runs_df'])
print('best experiment:', context['best_experiment'])
print('best test metrics:', context['best_test_metrics'])
context['gru_result']['config']

,experiment_id,task,dataset,seed,split_summary,window_size,horizon,model_summary,features_summary,scaler,optimizer,lr,epochs_trained,best_val_mae,best_val_rmse,best_val_mape,test_mae,test_rmse,test_mape,notes
0,B1,forecasting,S12-hw-dataset.csv,42,train=2025-01-01 00:00:00..2025-05-06 23:00:00...,,1,naive-last,last observed target,,,,,6.444815,8.201023,4.397922,,,,No training required
1,B2,forecasting,S12-hw-dataset.csv,42,train=2025-01-01 00:00:00..2025-05-06 23:00:00...,24,1,moving-average-24,rolling mean of target using past values only,,,,,13.397980,16.169904,9.198834,,,,No training required
2,B3,forecasting,S12-hw-dataset.csv,42,train=2025-01-01 00:00:00..2025-05-06 23:00:00...,,1,Ridge(alpha=1.0),"lag_1, lag_7, lag_14, lag_24, lag_168, rolling...","StandardScaler(features, fit on train)",,,,4.807380,6.163091,3.214340,4.484559,5.78157,2.886988,"Lag, rolling and calendar features without fut..."
3,R1,forecasting,S12-hw-dataset.csv,42,train=2025-01-01 00:00:00..2025-05-06 23:00:00...,48,1,"GRU(hidden_size=32, layers=1)",past target sequence only,StandardScaler(train target only),Adam,0.001,27,5.586523,7.298702,3.751106,,,,Best checkpoint selected by validation MAE


best experiment: B3
best test metrics: {'mae': 4.48455939635741, 'rmse': 5.781570004437078, 'mape': 2.8869875964630007}


{'seed': 42,
 'window_size': 48,
 'hidden_size': 32,
 'batch_size': 64,
 'learning_rate': 0.001,
 'max_epochs': 40,
 'optimizer': 'Adam',
 'loss': 'MSELoss',
 'scaler': 'StandardScaler(train target only)',
 'device': 'cpu',
 'best_epoch': 27}

В `GRU` масштабирование выполняется только по `train`, лучшая версия модели выбирается по `validation MAE`, а затем сохраняется в `artifacts/best_gru.pt`.

## 5. Итоговые артефакты и визуализации

In [6]:
fig_paths = [
    paths.figures_dir / 'series_split.png',
    paths.figures_dir / 'baselines_compare.png',
    paths.figures_dir / 'gru_learning_curves.png',
    paths.figures_dir / 'best_forecast_test.png',
]
for fig_path in fig_paths:
    img = plt.imread(fig_path)
    plt.figure(figsize=(14, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(fig_path.name)
    plt.show()

pd.read_csv(paths.runs_path)

/tmp/ipykernel_53737/3430015120.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,experiment_id,task,dataset,seed,split_summary,window_size,horizon,model_summary,features_summary,scaler,optimizer,lr,epochs_trained,best_val_mae,best_val_rmse,best_val_mape,test_mae,test_rmse,test_mape,notes
0,B1,forecasting,S12-hw-dataset.csv,42,train=2025-01-01 00:00:00..2025-05-06 23:00:00...,NaN,1,naive-last,last observed target,NaN,NaN,NaN,NaN,6.444815,8.201023,4.397922,NaN,NaN,NaN,No training required
1,B2,forecasting,S12-hw-dataset.csv,42,train=2025-01-01 00:00:00..2025-05-06 23:00:00...,24.0,1,moving-average-24,rolling mean of target using past values only,NaN,NaN,NaN,NaN,13.397980,16.169904,9.198834,NaN,NaN,NaN,No training required
2,B3,forecasting,S12-hw-dataset.csv,42,train=2025-01-01 00:00:00..2025-05-06 23:00:00...,NaN,1,Ridge(alpha=1.0),"lag_1, lag_7, lag_14, lag_24, lag_168, rolling...","StandardScaler(features, fit on train)",NaN,NaN,NaN,4.807380,6.163091,3.214340,4.484559,5.78157,2.886988,"Lag, rolling and calendar features without fut..."
3,R1,forecasting,S12-hw-dataset.csv,42,train=2025-01-01 00:00:00..2025-05-06 23:00:00...,48.0,1,"GRU(hidden_size=32, layers=1)",past target sequence only,StandardScaler(train target only),Adam,0.001,27.0,5.586523,7.298702,3.751106,NaN,NaN,NaN,Best checkpoint selected by validation MAE


Лучшая модель выбирается строго по `validation`, а `test` используется только один раз для финальной оценки победителя. Это защищает эксперимент от подгонки под отложенную выборку и делает сравнение честным.